> Organized from the original notebook. Saved outputs are historical and have not been regenerated. See `docs/analysis.md` before interpreting results.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "project_paths.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Launch Jupyter inside the Cancer-overview-ml project.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from project_paths import *
ensure_directories()


In [37]:
import numpy as np
import pandas as pd
import glob
import os
from collections import Counter

In [38]:
testfile_dir = str(CLEANED_TEXT_DIR)
textfilesList = glob.glob(os.path.join(testfile_dir, "*.txt"))
totalDocuments = len(textfilesList)
uni = str(NGRAM_DIR / "goodUni_cleaned.csv")
bi = str(NGRAM_DIR / "goodbi_cleaned.csv")
tri = str(NGRAM_DIR / "goodTriGrams.csv")
quad = str(NGRAM_DIR / "goodQuadgram.csv")

In [39]:
def extractGram(csv_file, n):
    gram = []
    file = pd.read_csv(csv_file)
    column_name = f'{n}-gram'  
    if column_name in file.columns:
        for term in file[column_name]:
            gram.append(term)
    return gram

In [40]:
uni_gram_li = extractGram(uni, 1)
bi_gram_li = extractGram(bi, 2) 
tri_gram_li = extractGram(tri, 3)
quad_gram_li = extractGram(quad, 4)

In [41]:
all_grams = uni_gram_li + bi_gram_li + tri_gram_li + quad_gram_li
print({len(all_grams)})

{37968}


In [ ]:
def count_ngrams_in_doc(doc_text, vocab):
    counts = Counter()
    lowered_text = doc_text.lower()
    for phrase in vocab:
        phrase_str = str(phrase).strip().lower()
        if phrase_str:
            count = lowered_text.count(phrase_str)
            if count > 0:
                counts[phrase_str] = count
    return counts


In [43]:
doc_word_counts = []
for filepath in textfilesList:
    with open(filepath, 'r', encoding='utf-8') as infile:
        doc_text = infile.read()
        # Count exact phrase occurrences for all vocab terms (uni/bi/tri/quad)
        counts = count_ngrams_in_doc(doc_text, all_grams)
        doc_word_counts.append(counts)

In [44]:
cooccurr = []
for phrase in all_grams:
    tempDict = {'name': phrase}
    for i, counts in enumerate(doc_word_counts):
        tempDict[i] = counts.get(phrase, 0)
    cooccurr.append(tempDict)

In [45]:
cooccur_df = pd.DataFrame(cooccurr)

In [46]:
cooccur_df

,name,0,1,2,3,4,5,6,7,8,...,2275,2276,2277,2278,2279,2280,2281,2282,2283,2284
0,cancer,7,38,57,7,7,7,7,7,25,...,42,53,38,6,24,20,76,16,23,21
1,tumor,0,5,1,0,0,0,0,0,16,...,1,2,3,0,2,1,5,0,0,0
2,cell,0,6,1,0,0,0,0,0,15,...,1,7,5,0,4,2,11,0,4,0
3,therapy,0,5,4,0,0,0,0,0,10,...,0,11,10,0,3,1,8,0,3,1
4,drug,1,0,1,1,1,1,1,1,18,...,0,1,4,2,2,1,0,0,25,23
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37963,detection genetic medicine cancer,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
37964,prevention detection genetic medicine,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
37965,cancer prevention detection genetic,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
37966,late cancer prevention detection,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [47]:
cooccur_df.to_csv(MATRIX_FILE,index=False)